In [ ]:
import numpy as np
import pandas as pd
import time
import os
import scipy.io as sio
import matplotlib.pyplot as plt
import math
from scipy.special import logsumexp, gammainc
from scipy import stats
from scipy.stats import ncx2
import random

In [ ]:
# Compute the normalizing constant when d or sigmas change
def compute_log_Z_even(y, sigmas):
  """Computes log Z and Z for even dimensions d in {2, 4, 6, 8}
  with prior eta(x) on ||x|| <= 1, weights=(0.1, 0.9), and sigmas=(0.1, 0.01).
  """
  y = np.asarray(y, dtype=float)
  d = len(y)
  if d not in {2, 4, 6, 8, 10}:
    raise ValueError(f"Expected d in {{2, 4, 6, 8, 10}}, got d={d}")

  m = d // 2 # only take the integer part
  weights = [0.1, 0.9]

  # log V_d = log(pi^m / m!)
  log_V_d = m * np.log(np.pi) - math.lgamma(m + 1)
  norm_y_sq = np.sum(y**2)

  log_terms = []
  for w, sigma in zip(weights, sigmas):
    lam = norm_y_sq / (sigma**2)
    x_val = 1.0 / (sigma**2)
    log_cdf = ncx2.logcdf(x_val, df=d, nc=lam)
    log_terms.append(np.log(w) - log_V_d + log_cdf)

  log_Z = logsumexp(log_terms)
  return log_Z, np.exp(log_Z)

In [ ]:
def simprior_fn(N, options):
    """
    Function to simulate from the prior.
    
    Parameters:
    N       : int, sample size
    options : object with attribute `d` (dimension)
    
    Returns:
    samples : ndarray of shape (N, d)
    """
    d = options.d
    
    samples = np.zeros((N, d))
    
    for i in range(N):
        samples[i, :] = bespoke_exact(1, d)
        
    return samples

In [ ]:
def logprior_fn(x, options):
    """
    Evaluates the log prior for a given sample.
    """
    if np.linalg.norm(x) < 1:
        logprior = 0.0
    else:
        logprior = -np.inf
        
    return logprior

In [ ]:
def loglike_fn(x, options):
    """
    Gets the log likelihood (Gaussian mixture pdf) for parameter x
    """
    d = options.d
    u = 0.1
    v = 0.01
    
    a1 = 0.1
    a2 = 0.9
    
    # Ensure x is a numpy array for element-wise squaring
    x = np.asarray(x)
    s = np.sum(x**2)
    
    c1 = np.log(a1) - (d / 2) * np.log(2 * np.pi * u**2) - 0.5 * s / u**2
    c2 = np.log(a2) - (d / 2) * np.log(2 * np.pi * v**2) - 0.5 * s / v**2
    
    mxm = max(c1, c2)
    mnm = min(c1, c2)
    
    loglike = mxm + np.log(1 + np.exp(mnm - mxm))
    
    return loglike

In [ ]:
def get_ids(L, U, alpha):
    """
    Returns the indices of the couples (L^k, U^k) lying above and below
    some order statistic.
    
    Parameters:
    L     : ndarray, likelihood values
    U     : ndarray, uniform random numbers
    alpha : float, adaptation parameter
    
    Returns:
    threshL   : the new threshold level for L 
    el_ind    : boolean array of whether sample k is strictly above the threshold
    under_ind : the indices of samples underneath threshold or equal to it
    over_ind  : the indices of samples over the threshold
    M         : Sorted array of [original_id, L, U]
    """
    L = np.asarray(L)
    U = np.asarray(U)
    N = len(L)
    
    assert len(L) == len(U), "L and U must have the same length"
    
    # Calculate the number of elements in the 'under' group
    ostat_id = int(np.floor(N * (1 - alpha)))
    
    # np.lexsort sorts by the last given array first (L), and breaks ties using the preceding arrays (U). 
    final_order = np.lexsort((U, L))
    
    # Split the indices based on the order statistic
    under_ind = final_order[:ostat_id]
    over_ind = final_order[ostat_id:]
    
    # The threshold is the highest 'L' value in the 'under' group
    threshL = L[final_order[ostat_id - 1]]
    
    # Create the boolean mask for the elite indices
    el_ind = np.zeros(N, dtype=bool)
    el_ind[over_ind] = True
    
    # Construct the M matrix (using 0-based indices for the first column)
    M = np.column_stack((final_order, L[final_order], U[final_order]))
    
    return threshL, el_ind, under_ind, over_ind, M

In [ ]:
def compute_ESS_diff(thing, gammavar, loglike, desired):
    """
    Computes the difference between the current Effective Sample Size (ESS) 
    and the desired ESS (used for bisection root-finding).
    """
    # Ensure loglike is a numpy array for element-wise operations
    loglike = np.asarray(loglike)
    
    # log of unnormalized weights
    weight = (thing - gammavar) * loglike
    # numerical stability before doing exponential
    weight = weight - np.max(weight)
    # convert back to the unnormalized weights
    weight = np.exp(weight)
    # derive normalized weights
    weight = weight / np.sum(weight)
    
    result = 1.0 / np.sum(weight**2) - desired
    
    return result

In [ ]:
def bisection(f, a, b):
    """
    Solves the equation f(x) = 0 using the bisection method.
    
    Parameters:
    f : callable, the function to find the root for
    a : float, lower bound of the interval
    b : float, upper bound of the interval
    
    Returns:
    p : float, the estimated root
    """
    if f(a) * f(b) > 0:
        print('Wrong choice bro')
        return None
    else:
        p = (a + b) / 2.0
        err = abs(f(p))
        
        while err > 1e-7: # stopping criteria
            if f(a) * f(p) < 0:
                b = p
            else:
                a = p
                
            p = (a + b) / 2.0
            err = abs(f(p))
            
        return p

In [ ]:
def bespoke_exact(dist, d):
    """
    Generates a point closer than distance 'dist' from the origin
    i.e., generate point in d-sphere of radius 'dist'
    """
    # Generate standard normal vector
    Y = np.random.randn(d)
    
    # Generate uniform random variable
    U = np.random.rand()
    
    # Calculate radius scaling factor
    R = dist * (U ** (1.0 / d))
    
    # Normalize Y to unit sphere and scale by R
    X = R * (Y / np.linalg.norm(Y))
    
    return X

In [ ]:
# NS 
def NS_bespoke_exact(loglike_fn, simprior_fn, options, verbose=False):
    """
    NS with bespoke, exact sampling for the move steps.
    
    Parameters:
    loglike_fn  : callable, log likelihood function taking (sample, options)
    simprior_fn : callable, prior simulation taking (N, options)
    options     : object with attributes `N` and `desired_count`
                  - options.N: Size of population of particles
                  - options.desired_count: the desired number of likelihood evaluations (for use in the stopping rule instead of stopping_epsilon)
    verbose     : bool, set to True to get running updates
    
    Returns:
    log_evidence      : The log evidence estimate from vanilla NS
    log_evidence_star : The log evidence estimate from NS*
    count_loglike     : The total log likelihood computations
    """
    # Number particles will be drawn
    N = options.N
    
    # Draw N particles from the prior
    theta_curr = simprior_fn(N, options)
    # Get the dimension of the parameter space (i.e., d=10 for the spike-and-slab example)
    d = theta_curr.shape[1]
    
    # Initialise normalizing constants for log evidence estimates 
    log_evidence = -np.inf  
    log_evidence_star = -np.inf
    # Tracking the initial state
    # Counts the sequential likelihood threshold steps
    t = 1
    
    loglike_curr = np.zeros(N)
    # Compute initial log likelihoods for all particles 
    for i in range(N):
        loglike_curr[i] = loglike_fn(theta_curr[i, :], options)

    # Initialize the count of log likelihood evaluations (i.e., N evaluations for the initial particles)
    count_loglike = N
    # Stopping condition: continue until the total number of log likelihood evaluations reaches the desired count
    while count_loglike <= options.desired_count:
        # Track progressive shell slicing
        t += 1
        # Find the minimum log likelihood and its index
        levels = np.min(loglike_curr) # the minimum log likelihood value = log(L_t^N)
        min_loc = np.argmin(loglike_curr) # the index of the minimum log likelihood = m
        
        # --- log \tilde{w}_{t-1} = log(L_t^N) + log(alpha_{t-1}-alpha_t) ---
        # ---- NS with alpha_t = exp(-t/N) (Algorithm 1) ----
        log_weight = levels - t/N + np.log(np.exp(1/N) - 1)
        # Increment of log(Z^{N, T^N})
        log_evidence = np.logaddexp(log_evidence, log_weight) 

        # ---- NS with alpha^t = ((N-1)/N)^t (Equation 27)----
        log_weight_star = levels + (t - 1) * np.log(N - 1) - t * np.log(N)
        # Increment of log(Z^{N, T^N})
        log_evidence_star = np.logaddexp(log_evidence_star, log_weight_star)
        
        if verbose:
            print(f"\nIter {t}\tLevel: {levels:.4f}\n\t\tCurrent log Z: {log_evidence:.4f}\n")

        # Compute the Euclidean distance (L2 norm) between the sample with the minimum log likelihood from the origin    
        dists = np.linalg.norm(theta_curr[min_loc, :])
        
        # Replace worst-performing particle with new sample which is generated by the prior (eta) 
        # (the uniform distribution on a sphere of radius 'dists')
        theta_curr[min_loc, :] = bespoke_exact(dists, d)
        # Recompute the log likelihood for the newly sampled particle
        loglike_curr[min_loc] = loglike_fn(theta_curr[min_loc, :], options)
        count_loglike += 1

    # Ensure the final remainder calculation uses the exact ending step count T^N
    t += 1
    
    # Filling-in
    inner_logsumexp = logsumexp(loglike_curr - t/N - np.log(N))
    log_evidence = np.logaddexp(log_evidence, inner_logsumexp)
    
    inner_logsumexp_star = logsumexp(loglike_curr + t * np.log(N - 1) - (t + 1) * np.log(N))
    log_evidence_star = np.logaddexp(log_evidence_star, inner_logsumexp_star)
    
    return log_evidence, log_evidence_star, count_loglike

In [ ]:
# NS with Random Walk (RW) move steps
def NS_bespoke_RW(loglike_fn, logprior_fn, simprior_fn, options, verbose=False):
    """
    NS with Random Walk (RW) move steps.
    
    Parameters:
    loglike_fn  : callable, log likelihood function taking (sample, options)
    logprior_fn : callable, log prior function taking (sample, options)
    simprior_fn : callable, prior simulation taking (N, options)
    options     : object with attributes `N`, `R`, `sig`, and `desired_count`
                  - options.N: Size of population of particles
                  - options.R: Number of MCMC repeats per particle per SMC iteration
                  - options.sig: Standard deviation for mixture RW proposal (sig or sig/4)
                  - options.desired_count: the desired number of likelihood evaluations (for use in the stopping rule instead of stopping_epsilon) 
    verbose     : bool, set to True to get running updates
    
    Returns:
    log_evidence      : The log evidence estimate from vanilla NS
    log_evidence_star : The log evidence estimate from NS*
    count_loglike     : The total log likelihood computations
    """
    # Number particles will be drawn
    N = options.N
    
    # Draw N particles from the prior
    theta_curr = simprior_fn(N, options)
    # Get the dimension of the parameter space (i.e., d=10 for the spike-and-slab example)
    d = theta_curr.shape[1]
    
    # Initialise normalizing constants for log evidence estimates
    log_evidence = -np.inf
    log_evidence_star = -np.inf
    # Tracking the initial state
    # Counts the sequential likelihood threshold steps
    t = 1
    # Number of random walk steps
    R = options.R
    
    # Compute initial log likelihoods and log priors for all particles
    loglike_curr = np.zeros(N)
    logprior_curr = np.zeros(N)
    for i in range(N):
        loglike_curr[i] = loglike_fn(theta_curr[i, :], options)
        logprior_curr[i] = logprior_fn(theta_curr[i, :], options)

    # Initialize the count of log likelihood evaluations (i.e., N evaluations for the initial particles)
    count_loglike = N
    
    while count_loglike <= options.desired_count:
        # Track progressive shell slicing
        t += 1
        
        # Find the minimum log likelihood and its index
        levels = np.min(loglike_curr) # the minimum log likelihood value = log(L_t^N)
        min_loc = np.argmin(loglike_curr) # the index of the minimum log likelihood = m
        
        # Same as in NS_bespoke_exact, compute the log weights for both NS and adjusted NS*
        log_weight = levels - t/N + np.log(np.exp(1/N) - 1)
        log_evidence = np.logaddexp(log_evidence, log_weight) 
        
        # adjusted version
        log_weight_star = levels + (t - 1) * np.log(N - 1) - t * np.log(N)
        log_evidence_star = np.logaddexp(log_evidence_star, log_weight_star)
        
        if verbose:
            print(f"\nIter {t}\tLevel: {levels:.4f}\n\t\tCurrent log Z: {log_evidence:.4f}\n")

        # Clone a random surviving particle (not the worst one) to replace the worst-performing particle
        choice = min_loc
        # Ensure that the chosen index for replacement is not the same as the index of the minimum log likelihood
        while choice == min_loc:
            choice = np.random.randint(0, N)

        # Replace the worst-performing particle with a copy of another randomly chosen particle (not the worst one)    
        theta_curr[min_loc, :] = theta_curr[choice, :]
        loglike_curr[min_loc] = loglike_curr[choice]
        logprior_curr[min_loc] = logprior_curr[choice]
        
        # Perform R random walk steps for the newly cloned particle
        # MCMC kernel: 
        # - The input options.sig = 0.1 later on
        # - The normal proposal standard deviation is either sig=1/10 or sig/4=1/40, chosen with equal probability
        for k in range(R):
            if np.random.rand() < 0.5:
                sig = options.sig
            else:
                sig = options.sig / 4
                
            # Copy the current particle (already been replaced) to propose a new one
            theta_prop = np.copy(theta_curr[min_loc, :]) 
            # Randomly select a dimension to perturb
            loc = np.random.randint(0, d)
            # Perturb the selected dimension with Gaussian noise
            theta_prop[loc] = theta_prop[loc] + sig * np.random.randn() 
            
            # Compute the log prior for the proposed particle 
            logprior_prop = logprior_fn(theta_prop, options)
            # Calculate the Metropolis-Hastings acceptance ratio
            # A standard random walk proposal is symmetric
            MH = np.exp(logprior_prop - logprior_curr[min_loc])
            
            if MH > np.random.rand():
                # If the proposed particle is accepted, compute its log likelihood
                loglike_prop = loglike_fn(theta_prop, options)
                # Increment the count of log likelihood evaluations for each proposed particle
                count_loglike += 1
                # If the proposed particle has a log likelihood greater than the current level, accept it
                if loglike_prop > levels:
                    theta_curr[min_loc, :] = theta_prop
                    loglike_curr[min_loc] = loglike_prop
                    logprior_curr[min_loc] = logprior_prop

    # Ensure the final remainder calculation uses the exact ending step count T^N
    t += 1
    
    # Filling-in
    inner_logsumexp = logsumexp(loglike_curr - t/N - np.log(N))
    log_evidence = np.logaddexp(log_evidence, inner_logsumexp)
    
    inner_logsumexp_star = logsumexp(loglike_curr + t * np.log(N - 1) - (t + 1) * np.log(N))
    log_evidence_star = np.logaddexp(log_evidence_star, inner_logsumexp_star)
    
    return log_evidence, log_evidence_star, count_loglike

In [ ]:
# NS-SMC with bespoke, exact sampling for the move steps
def NS_SMC_bespoke_exact(loglike_fn, simprior_fn, options, verbose=False):
    """
    NS-SMC with bespoke, exact sampling for the move steps.
    
    Parameters:
    loglike_fn  : callable, log likelihood function taking (sample, options)
    simprior_fn : callable, prior simulation taking (N, options)
    options     : object with attributes `N`, `levels`, `dists`, and `d`
                  - options.N: Size of population of particles
                  - options.levels: The fixed levels
                  - options.dists: Maximum norm of the sphere corresponding to levels
                  - options.d: Dimension of the parameter space
    verbose     : bool, set to True to get running updates
    
    Returns:
    log_evidence  : The log evidence estimate 
    count_loglike : The total log likelihood computations
    error_flag    : Zero if it finishes to level T, -t if stops because no particles 
                    are above the next threshold at level t, and t if it stops early 
                    because all particles are above the next threshold at level t.
    """
    # Number particles will be drawn
    N = options.N
    # The pre-determined array of thresholds
    levels = np.array(options.levels)
    # The maximum norm of the sphere corresponding to levels
    dists = np.array(options.dists)
    
    # Draw N particles from the prior
    theta_curr = simprior_fn(N, options)
    
    # Initialise
    log_evidence = -np.inf
    # The number of levels (thresholds) to be processed
    T = len(levels)
    
    # Initialise the product of the proportions of particles above the threshold at each level (P_0^N)
    prod_c = 1.0
    # Initialize the error flag to track the status of the algorithm
    error_flag = 0
    
    # Compute initial log likelihoods for all particles
    loglike_curr = np.zeros(N)
    for i in range(N):
        loglike_curr[i] = loglike_fn(theta_curr[i, :], options)

    # N evaluations for the initial particles  
    count_loglike = N
    
    # Count the sequential likelihood threshold steps
    t = 1 
    while t < T:
        # Find the indices of particles that are above the current threshold level
        el_ind = loglike_curr > levels[t]
        # Sum the number of particles that are above the threshold
        num_elites = np.sum(el_ind)
        # Calculate the proportion of elite particles (in the slice): 
        # c = (1/N)*sum(w_t^{(k)}) = (1/N)*sum(1{L > l_t}) 
        c = num_elites / N

        # If no particles are above the threshold and it's not the last level, set error_flag and jump to the end of the loop
        if c == 0 and t != (T - 1):
            error_flag = -(t + 1)
            t = T - 1  # Force jump to end of loop

        # If all particles are above the threshold, set error_flag to t+1    
        if c != 1:
            # --- weighted (unnormalized) samples ---
            # reweight eta_{n-1} towards the target distribution pi
            logw = loglike_curr[~el_ind] + np.log(prod_c) # Eq (18)
            # normalizing constant estimation for pi_{t-1}
            term = logsumexp(logw) - np.log(N)
            # increment normalizing constant for pi 
            log_evidence = np.logaddexp(log_evidence, term)
        else:
            # if all particles are above the threshold, set
            error_flag = t + 1

        # Update the product of the proportions of particles above the threshold for the next iteration 
        # Normalizing constant for eta_t   
        prod_c = prod_c * c
        
        if verbose:
            print(f"\nIter {t+1}\tLevel: {levels[t]:.4f}\n\t\tCurrent log Z: {log_evidence:.4f}\n\t\tTotal elite: {num_elites}\n")
            
        # Resample and Move
        if levels[t] != np.inf and num_elites > 0:
            # Note: since all valid weights are equal, we can skip explicit multinomial weight arrays 
            # and just sample uniformly with replacement directly from the elite indices
            
            # Identify the indices of the elite particles (those above the threshold)
            # elite_indices = np.where(el_ind)[0]
            # Resample, selecting N indices from the elite_indices with replacement
            # inds = np.random.choice(elite_indices, size=N, replace=True)
            
            # Update the current particles 
            # theta_curr = theta_curr[inds, :]
            # Update the log likelihoods to only include the resampled elite particles
            # loglike_curr = loglike_curr[inds]
            
            # All the weights are equal so we just sample uniformly
            for i in range(N):
                theta_curr[i, :] = bespoke_exact(dists[t], options.d)
                loglike_curr[i] = loglike_fn(theta_curr[i, :], options)
                count_loglike += 1
                
        t += 1
        
    return log_evidence, count_loglike, error_flag

In [ ]:
# NS-SMC with Random Walk (RW) move steps
def NS_SMC_bespoke_RW(loglike_fn, logprior_fn, simprior_fn, options, verbose=False):
    """
    NS-SMC with Random Walk (RW) move steps.
    
    Parameters:
    loglike_fn  : callable, log likelihood function taking (sample, options)
    logprior_fn : callable, log prior function taking (sample, options)
    simprior_fn : callable, prior simulation taking (N, options)
    options     : object with attributes `N`, `levels`, `R`, and `sig`
                  - options.N: Size of population of particles
                  - options.levels: The fixed levels
                  - options.R: Number of MCMC repeats per particle per SMC iteration
                  - options.sig: Standard deviation for mixture RW proposal (sig or sig/4)
    verbose     : bool, set to True to get running updates
    
    Returns:
    log_evidence  : The log evidence estimate 
    count_loglike : The total log likelihood computations
    error_flag    : Zero if it finishes to level T, -t if stops because no particles 
                    are above the next threshold at level t, and t if it stops early 
                    because all particles are above the next threshold at level t.
    """
    # Number particles will be drawn
    N = options.N
    # The pre-determined array of thresholds
    levels = np.array(options.levels)
    # Number of random walk steps
    R = options.R
    
    # Draw N particles from the prior
    theta_curr = simprior_fn(N, options)
    # Get the dimension of the parameter space (i.e., d=10 for the spike-and-slab example)
    d = theta_curr.shape[1]
    
    # Initialise
    log_evidence = -np.inf
    # The number of levels (thresholds) to be processed
    T = len(levels)
    
    # Initialise the product of the proportions of particles above the threshold at each level (P_0^N)
    prod_c = 1.0
    # Initialize the error flag to track the status of the algorithm
    error_flag = 0

    # Intialize for tracking
    log_unnormalized_w = []
    log_normalized_w = []
    log_evidence_t = []
    prop_particles_inside_spike_radius_1sd = []
    prop_particles_inside_spike_radius_2sd = []
    prop_particles_inside_spike_radius_3sd = []
    
    # Compute initial log likelihoods for all particles
    loglike_curr = np.zeros(N)
    for i in range(N):
        loglike_curr[i] = loglike_fn(theta_curr[i, :], options)

    # N evaluations for the initial particles     
    count_loglike = N
    accept_rate_t = []
    accept_spike_t = []
    accept_slab_t = []
    
    # Count the sequential likelihood threshold steps
    t = 1
    while t < T:
        prop_particles_inside_spike_radius_1sd.append(np.sum(np.all(np.abs(theta_curr) < 1 * 0.01, axis=1))/N)
        prop_particles_inside_spike_radius_2sd.append(np.sum(np.all(np.abs(theta_curr) < 2 * 0.01, axis=1))/N)
        prop_particles_inside_spike_radius_3sd.append(np.sum(np.all(np.abs(theta_curr) < 3 * 0.01, axis=1))/N)
        
        # Find the indices of particles that are above the current threshold level
        el_ind = loglike_curr > levels[t] # indices of elite set
        # Sum the number of particles that are above the threshold
        num_elites = np.sum(el_ind)
        # Calculate the proportion of elite particles (in the slice): 
        # c = (1/N)*sum(w_t^{(k)}) = (1/N)*sum(1{L > l_t}) 
        c = num_elites / N
        
        # If no particles are above the threshold and it's not the last level, set error_flag and jump to the end of the loop
        if c == 0 and t != (T - 1):
            error_flag = -(t + 1) 
            t = T - 1 
            
        if c != 1:
            # --- weighted (unnormalized) samples ---
            # reweight eta_{n-1} towards the target distribution pi
            logw = loglike_curr[~el_ind] + np.log(prod_c) # Eq (18)
            log_unnormalized_w.append(logw)
            # normalizing constant estimation for pi_{t-1}
            term = logsumexp(logw) - np.log(N)
            log_normalized_w.append(term)
            # increment normalizing constant for pi
            log_evidence = np.logaddexp(log_evidence, term)
            log_evidence_t.append(log_evidence)
        else:
            # if all particles are above the threshold, set
            error_flag = t + 1

        # Update the product of the proportions of particles above the threshold for the next iteration 
        # Normalizing constant for eta_t     
        prod_c = prod_c * c
        
        if verbose:
            print(f"\nIter {t+1}\tLevel: {levels[t]:.4f}\n\t\tCurrent log Z: {log_evidence:.4f}\n\t\tTotal elite: {num_elites}\n")
            
        # Resample and Move
        if levels[t] != np.inf and num_elites > 0:
            # Uniform resampling directly from elite indices (skipping multinomial array creation)
            # Identify the indices of the elite particles (those above the threshold)
            elite_indices = np.where(el_ind)[0]
            # Resample, selecting N indices from the elite_indices with replacement
            inds = np.random.choice(elite_indices, size=N, replace=True)
            
            # Need to evaluate log prior for the first time
            if t == 1: 
                logprior_curr = np.zeros(N)
                # Compute on unique indices *before* overwriting theta_curr
                for i in np.unique(inds):
                    logprior_curr[i] = logprior_fn(theta_curr[i, :], options)

            # Update the current particles and their log likelihoods and log priors to only include the resampled elite particles        
            theta_curr = theta_curr[inds, :]
            loglike_curr = loglike_curr[inds]
            logprior_curr = logprior_curr[inds]

            accept_N = []
            accept_spike_N = []
            accept_slab_N = []
            prop_spike_N = []   
            prop_slab_N = []
            # Perform R random walk steps for each particle in the resampled set
            for i in range(N):
                count_accept = 0
                count_accept_spike = 0
                count_accept_slab = 0
                prop_spike = 0 
                prop_slab = 0
                # MCMC kernel: 
                # - The input options.sig = 0.1 later on
                # - The normal proposal standard deviation is either sig=1/10 or sig/4=1/40, chosen with equal probability
                for k in range(R):
                    in_spike = np.linalg.norm(theta_curr[i, :]) < 3 * 0.01
                    if in_spike:
                        prop_spike += 1
                    else:
                        prop_slab += 1

                    if np.random.rand() < 0.5:
                        sig = options.sig
                    else:
                        sig = options.sig / 4

                    # Copy the current particle (already been replaced) to propose a new one    
                    theta_prop = np.copy(theta_curr[i, :])
                    # Randomly select a dimension to perturb
                    loc = np.random.randint(0, d)
                    # Perturb the selected dimension with Gaussian noise
                    theta_prop[loc] += sig * np.random.randn()
                    
                    # Compute the log prior for the proposed particle
                    logprior_prop = logprior_fn(theta_prop, options)
                    
                    # Calculate the Metropolis-Hastings acceptance ratio
                    # A standard random walk proposal is symmetric
                    if np.exp(logprior_prop - logprior_curr[i]) > np.random.rand():
                        # If the proposed particle is accepted, compute its log likelihood
                        loglike_prop = loglike_fn(theta_prop, options)
                        # Increment the count of log likelihood evaluations for each proposed particle
                        count_loglike += 1
                        
                        # If the proposed particle has a log likelihood greater than the current level, accept it 
                        # and update the current particle
                        if loglike_prop > levels[t]:
                            theta_curr[i, :] = theta_prop
                            loglike_curr[i] = loglike_prop
                            logprior_curr[i] = logprior_prop
                            count_accept += 1
                            
                            if in_spike:
                                count_accept_spike += 1
                            else: 
                                count_accept_slab += 1
                            
                accept_N.append(count_accept)
                accept_spike_N.append(count_accept_spike)
                accept_slab_N.append(count_accept_slab)
                prop_spike_N.append(prop_spike) 
                prop_slab_N.append(prop_slab)
                            
            total_prop_spike = np.sum(prop_spike_N)
            total_prop_slab = np.sum(prop_slab_N)

            accept_rate_t.append(np.sum(accept_N) / (R * N)) 
            accept_spike_t.append(np.sum(accept_spike_N) / total_prop_spike if total_prop_spike > 0 else np.nan) 
            accept_slab_t.append(np.sum(accept_slab_N) / total_prop_slab if total_prop_slab > 0 else np.nan) 
                            
        t += 1
    prop_particles_inside_spike_radius_1sd.append(np.sum(np.all(np.abs(theta_curr) < 1 * 0.01, axis=1))/N)
    prop_particles_inside_spike_radius_2sd.append(np.sum(np.all(np.abs(theta_curr) < 2 * 0.01, axis=1))/N)
    prop_particles_inside_spike_radius_3sd.append(np.sum(np.all(np.abs(theta_curr) < 3 * 0.01, axis=1))/N)
        
    return log_evidence, count_loglike, error_flag, log_unnormalized_w, log_normalized_w, np.array(log_evidence_t), np.array(prop_particles_inside_spike_radius_1sd), np.array(prop_particles_inside_spike_radius_2sd), np.array(prop_particles_inside_spike_radius_3sd), np.array(accept_rate_t), np.array(accept_spike_t), np.array(accept_slab_t), total_prop_spike, total_prop_slab

In [ ]:
# TA-SMC with Random Walk (RW) move steps
def TA_SMC_bespoke_RW(loglike_fn, logprior_fn, simprior_fn, options, verbose=False):
    """
    ATA-SMC with RW move steps.
    
    Parameters:
    loglike_fn  : callable, log likelihood function taking (sample, options)
    logprior_fn : callable, log prior function taking (sample, options)
    simprior_fn : callable, prior simulation taking (N, options)
    options     : object with attributes `N`, `gammavar`, `R`, and `sig`
                  - options.N: Size of population of particles
                  - options.gammavar: The fixed temperatures
                  - options.R: fixed number of MCMC repeats
                  - options.sig: Standard deviation for mixture RW proposal (sig or sig/4)
    verbose     : bool, set to True to get running updates
    
    Returns:
    log_evidence  : The log evidence estimate
    count_loglike : The total log likelihood computations
    """
    # Total number of particles will be drawn
    N = options.N
    # The fixed temperatures
    gammavar = np.array(options.gammavar)
    # Number of random walk steps
    R = options.R
    
    # Draw N particles from the prior
    theta = simprior_fn(N, options)
    # Get the dimension of the parameter space (i.e., d=10 for the spike-and-slab example)
    d = theta.shape[1]
    
    # Initialise
    log_evidence = 0.0
    # The number of temperatures to be processed
    T = len(gammavar)
    
    # Compute initial log likelihoods and priors for all particles
    loglike = np.zeros(N)
    logprior = np.zeros(N)
    for i in range(N):
        loglike[i] = loglike_fn(theta[i, :], options)
        logprior[i] = logprior_fn(theta[i, :], options) # some of these will be wasted - maybe move to later

    # N evaluations for the initial particles      
    count_loglike = N
    
    for t in range(T - 1):
        # --- Reweight ---
        # Incremental LOG of unnormalized weights
        w = (gammavar[t + 1] - gammavar[t]) * loglike
        # Update the normalizing constant at the current time
        log_evidence = log_evidence + logsumexp(w) - np.log(N)
        # Compute normalized weights for resample
        w = np.exp(w - logsumexp(w))
        
        if verbose:
            print(f"\nIter {t+1}\tInverse temperature: {gammavar[t+1]:.4f}\n\t\tCurrent log Z: {log_evidence:.4f}\n")

        # --- Resample ---    
        # Sampling with replacement according to weights
        ind = np.random.choice(N, size=N, replace=True, p=w)
        # Update the current theta, the log likelihoods and priors
        theta = theta[ind, :]
        loglike = loglike[ind]
        logprior = logprior[ind]
        
        # --- Mutate ---
        for i in range(N):
            for k in range(R):
                if np.random.rand() < 0.5:
                    sig = options.sig
                else:
                    sig = options.sig / 4.0
                    
                theta_prop = np.copy(theta[i, :])
                loc = np.random.randint(0, d)
                theta_prop[loc] += sig * np.random.randn()
                
                loglike_prop = loglike_fn(theta_prop, options)
                logprior_prop = logprior_fn(theta_prop, options)
                
                # Compute the log acceptance rate
                log_mh = gammavar[t + 1] * (loglike_prop - loglike[i]) + logprior_prop - logprior[i]
                # since log pi_{t+1}(x) = gamma_{t+1} log L(x) + log eta(x) + const
                # log_mh = log pi_{t+1}(x_prop) - log pi_{t+1}(x_i)
                
                if np.exp(log_mh) > np.random.rand():
                    theta[i, :] = theta_prop
                    loglike[i] = loglike_prop
                    logprior[i] = logprior_prop
                    
        count_loglike += N * R
        
    return log_evidence, count_loglike

In [ ]:
# ATA-SMC with Random Walk (RW) move steps
def ATA_SMC_bespoke_RW(loglike_fn, logprior_fn, simprior_fn, options, verbose=False):
    """
    ATA-SMC with RW move steps.
    
    Parameters:
    loglike_fn  : callable, log likelihood function taking (sample, options)
    logprior_fn : callable, log prior function taking (sample, options)
    simprior_fn : callable, prior simulation taking (N, options)
    options     : object with attributes `N`, `alpha`, `R`, and `sig`
                  - options.N: Size of population of particles
                  - options.alpha: Threshold so ESS at each step is alpha*N
                  - options.R: Number of MCMC repeats per particle per SMC iteration
                  - options.sig: Standard deviation for mixture RW proposal (sig or sig/4)
    verbose     : bool, set to True to get running updates
    
    Returns:
    log_evidence  : The log evidence estimate
    count_loglike : The total log likelihood computations
    gammavar      : The inverse temperatures for the likelihood annealing schedule
    """
    N = options.N
    alpha = options.alpha
    R = options.R

    # Intialize for tracking
    ess = []
    log_unnormalized_w = []
    normalized_w = []
    prop_particles_inside_spike_radius_1sd = []
    prop_particles_inside_spike_radius_2sd = []
    prop_particles_inside_spike_radius_3sd = []
    log_evidence_t = []
    
    # Draw N particles from prior
    theta = simprior_fn(N, options)
    d = theta.shape[1]
    
    # initialise
    gammavar = [0.0]
    log_evidence = 0.0
    t = 0 

    # Compute log likelihoods and priors of the intial N particles
    loglike = np.zeros(N)
    logprior = np.zeros(N)
    for i in range(N):
        loglike[i] = loglike_fn(theta[i, :], options)
        logprior[i] = logprior_fn(theta[i, :], options) # some of these will be wasted - maybe move to later
        
    count_loglike = N # N evaluations for intial particles
    accept_rate_t = []
    accept_spike_t = []
    accept_slab_t = []

    while gammavar[t] != 1.0: 
        # Check how many particles among N particles lies inside the spike radius
        prop_particles_inside_spike_radius_1sd.append(np.sum(np.all(np.abs(theta) < 1 * 0.01, axis=1))/N) # sd of the second component is 0.01
        prop_particles_inside_spike_radius_2sd.append(np.sum(np.all(np.abs(theta) < 2 * 0.01, axis=1))/N) 
        prop_particles_inside_spike_radius_3sd.append(np.sum(np.all(np.abs(theta) < 3 * 0.01, axis=1))/N) 
        # --- Finding gamma_{t+1} ---
        # Testing gammavar=1
        # Log of unnormalized weights if the next gamma is 1
        w = (1.0 - gammavar[t]) * loglike
        # Log of normalized weights
        w = w - logsumexp(w)
        ESS1 = np.exp(-logsumexp(2 * w))
        
        if ESS1 > alpha * N:
            gammavar.append(1.0)
        else:
            # define function f to perform bisection to find the next gamma
            f_target = lambda thing: compute_ESS_diff(thing, gammavar[t], loglike, alpha * N)
            # find the root gamma s.t. ESS(gamma)-alpha*N = 0
            next_gamma = bisection(f_target, gammavar[t], 1.0)
            gammavar.append(next_gamma)

        # --- Reweight ---
        # Incremental of log unnormalized weights between step t and t+1
        w = (gammavar[t + 1] - gammavar[t]) * loglike
        log_unnormalized_w.append(w)
        # Update the normalizing constant at the current time 
        log_evidence = log_evidence + logsumexp(w) - np.log(N)
        log_evidence_t.append(log_evidence)
        # Compute and append ESS
        ess.append(np.exp(-logsumexp(2 * (w - logsumexp(w))))) 
        # Normalized weights between step t and t+1
        w = np.exp(w - logsumexp(w))
        normalized_w.append(w)
        
        if verbose:
            print(f"\nIter {t+1}\tInverse temperature: {gammavar[t+1]:.4f}\n\t\tCurrent log Z: {log_evidence:.4f}\n")

        # --- Resample ---    
        # Sampling with replacement according to weights
        ind = np.random.choice(N, size=N, replace=True, p=w)
        # Update the current theta, the log likelihoods and priors
        theta = theta[ind, :]
        loglike = loglike[ind]
        logprior = logprior[ind]
        
        # --- Mutate ---
        accept_R = []
        accept_spike_R = []
        accept_slab_R = []
        prop_spike_R = []   
        prop_slab_R = []
        
        for k in range(R):
            count_accept = 0
            count_accept_spike = 0
            count_accept_slab = 0
            prop_spike = 0 
            prop_slab = 0
            
            for i in range(N):
                in_spike = np.linalg.norm(theta[i, :]) < 3 * 0.01
                if in_spike:
                    prop_spike += 1
                else:
                    prop_slab += 1

                if np.random.rand() < 0.5:
                    sig = options.sig
                else:
                    sig = options.sig / 4.0
                    
                theta_prop = np.copy(theta[i, :])
                loc = np.random.randint(0, d)
                theta_prop[loc] += sig * np.random.randn()
                
                loglike_prop = loglike_fn(theta_prop, options)
                logprior_prop = logprior_fn(theta_prop, options)
                
                # Compute the log acceptance rate
                log_mh = gammavar[t + 1] * (loglike_prop - loglike[i]) + logprior_prop - logprior[i]
                
                if np.exp(log_mh) > np.random.rand():
                    theta[i, :] = theta_prop
                    loglike[i] = loglike_prop
                    logprior[i] = logprior_prop
                    count_accept += 1

                    if in_spike:
                        count_accept_spike += 1
                    else: 
                        count_accept_slab += 1

            accept_R.append(count_accept)
            accept_spike_R.append(count_accept_spike)
            accept_slab_R.append(count_accept_slab)
            prop_spike_R.append(prop_spike) 
            prop_slab_R.append(prop_slab)
            
        total_prop_spike = np.sum(prop_spike_R)
        total_prop_slab = np.sum(prop_slab_R)
            
        accept_rate_t.append(np.sum(accept_R) / (R * N)) 
        accept_spike_t.append(np.sum(accept_spike_R) / total_prop_spike if total_prop_spike > 0 else np.nan) 
        accept_slab_t.append(np.sum(accept_slab_R) / total_prop_slab if total_prop_slab > 0 else np.nan)           
        count_loglike += N * R
        t += 1

    prop_particles_inside_spike_radius_1sd.append(np.sum(np.all(np.abs(theta) < 1 * 0.01, axis=1))/N)
    prop_particles_inside_spike_radius_2sd.append(np.sum(np.all(np.abs(theta) < 2 * 0.01, axis=1))/N)
    prop_particles_inside_spike_radius_3sd.append(np.sum(np.all(np.abs(theta) < 3 * 0.01, axis=1))/N)
        
    return log_evidence, count_loglike, np.array(gammavar), np.array(ess), np.array(log_unnormalized_w), np.array(normalized_w), np.array(prop_particles_inside_spike_radius_1sd), np.array(prop_particles_inside_spike_radius_2sd), np.array(prop_particles_inside_spike_radius_3sd), np.array(log_evidence_t), np.array(accept_rate_t), np.array(accept_spike_t), np.array(accept_slab_t), total_prop_spike, total_prop_slab

In [ ]:
# Diagnostic plot for ATA-SMC
def plot_ata_smc_diagnostics(
    *,
    gammavar,
    log_evidence_t,
    accept_rate_t,
    accept_spike_t,
    accept_slab_t,
    prop_1sd,
    prop_2sd,
    prop_3sd,
    normalized_w,
    ess,
    true_log_z=np.log(0.3921),
    label_prefix="ATA-SMC",
    max_weight_boxes=None,
):
    gammavar = np.asarray(gammavar)
    log_evidence_t = np.asarray(log_evidence_t)
    ess = np.asarray(ess)
    prop_1sd = np.asarray(prop_1sd)
    prop_2sd = np.asarray(prop_2sd)
    prop_3sd = np.asarray(prop_3sd)

    # 1. Running log Z over iterations
    plt.figure(figsize=(10, 4.5))
    plt.plot(log_evidence_t, linewidth=1.5, label=r"Running $\log \hat{Z}_t$")
    plt.axhline(
        true_log_z,
        color="r",
        linestyle="--",
        linewidth=1.5,
        label=rf"Ref $\log Z \approx {true_log_z:.5f}$",
    )
    plt.xlabel("Iteration (t)")
    plt.ylabel("Log Evidence")
    plt.title(f"{label_prefix}: Running Log Evidence over Iterations")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # 2. Running log Z vs. gamma_t
    gamma_x = gammavar[1:] if len(gammavar) == len(log_evidence_t) + 1 else gammavar
    plt.figure(figsize=(10, 4.5))
    plt.plot(gamma_x, log_evidence_t, linewidth=1.5, label=r"Running $\log \hat{Z}_t$")
    plt.axhline(
        true_log_z,
        color="r",
        linestyle="--",
        linewidth=1.5,
        label=rf"Ref $\log Z \approx {true_log_z:.5f}$",
    )
    plt.xlabel(r"$\gamma_t$")
    plt.ylabel("Log Evidence")
    plt.title(r"{}: Running Log Evidence vs. $\gamma_t$".format(label_prefix))
    plt.xlim(-0.02, 1.02)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # 3. Acceptance rates over iterations
    plt.figure(figsize=(10, 4.5))
    plt.plot(np.asarray(accept_rate_t) * 100, color="black", linewidth=1.8, label="Overall")
    plt.plot(
        np.asarray(accept_spike_t) * 100,
        color="tab:red",
        linestyle="--",
        linewidth=1.5,
        label="Within 3 SD of Spike",
    )
    plt.plot(
        np.asarray(accept_slab_t) * 100,
        color="tab:blue",
        linestyle="-.",
        linewidth=1.5,
        label="Beyond 3 SD (Slab)",
    )
    plt.xlabel("Iteration (t)")
    plt.ylabel("Acceptance Rate (%)")
    plt.title(f"{label_prefix}: Acceptance Rates across Iterations")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # 4. Effective sample size over iterations
    plt.figure(figsize=(10, 4.5))
    plt.plot(ess, marker="o", markersize=3.5, linewidth=1.5)
    plt.xlabel("Iteration (t)")
    plt.ylabel("ESS")
    plt.title(f"{label_prefix}: Effective Sample Size across Iterations")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.show()

    # 5. Tempering schedule gamma_t
    plt.figure(figsize=(10, 4.5))
    plt.plot(gammavar, linewidth=1.5)
    plt.xlabel("Step Index")
    plt.ylabel(r"$\gamma_t$")
    plt.title(f"{label_prefix}: Tempering Schedule")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.show()

    # 6. Tempering step size delta gamma_t
    plt.figure(figsize=(10, 4.5))
    plt.plot(np.diff(gammavar), marker="o", markersize=3.5, linewidth=1.5)
    plt.xlabel("Iteration (t)")
    plt.ylabel(r"$\Delta \gamma_t$")
    plt.title(f"{label_prefix}: Adaptive Tempering Step Size Over Iterations")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.show()

    # 7. Proportions inside 1, 2, 3 SD over iterations
    plt.figure(figsize=(10, 4.5))
    plt.plot(prop_1sd, label="1 SD", linewidth=1.5, color="tab:blue")
    plt.plot(prop_2sd, label="2 SD", linestyle="--", linewidth=1.5, color="tab:orange")
    plt.plot(prop_3sd, label="3 SD", linestyle="-.", linewidth=1.5, color="tab:green")
    plt.xlabel("Iteration (t)")
    plt.ylabel("Proportion of Particles")
    plt.title(f"{label_prefix}: Proportion Inside Spike Radius Over Iterations")
    plt.ylim(-0.02, 1.02)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # 8. Normalized weights boxplot across all iterations
    w_arr = np.asarray(normalized_w)
    n_steps = w_arr.shape[0]
    if max_weight_boxes is None or max_weight_boxes >= n_steps:
        indices = np.arange(n_steps)
    else:
        indices = np.unique(np.linspace(0, n_steps - 1, max_weight_boxes, dtype=int))

    plt.figure(figsize=(16, 4.5))
    plt.boxplot([np.asarray(w_arr[idx]).ravel() for idx in indices], showfliers=False)

    tick_step = max(1, len(indices) // 25)
    tick_positions = np.arange(1, len(indices) + 1, tick_step)
    plt.xticks(tick_positions, indices[tick_positions - 1] + 1, rotation=45)

    plt.xlabel("Iteration (t)")
    plt.ylabel("Normalized Weight")
    plt.title(f"{label_prefix}: Normalized Weights Across All {len(indices)} Iterations")
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

In [ ]:
# ANS-SMC
def ANS_SMC_bespoke_exact(loglike_fn, simprior_fn, options, verbose=False):
    """
    ANS-SMC with bespoke, exact sampling for the move steps.
    
    Parameters:
    loglike_fn  : callable, log likelihood function taking (sample, options)
    simprior_fn : callable, prior simulation taking (N, options)
    options     : object with attributes `N`, `alpha`, and `d`
                  - options.N: Size of population of particles
                  - options.alpha: Threshold so (1-alpha)*N samples are above the next threshold
                  - options.d: Dimension of the parameter space
    verbose     : bool, set to True to get running updates
    
    Returns:
    log_evidence  : The log evidence estimate
    count_loglike : The total log likelihood computations
    levels        : The levels for nested sampling
    dists         : Maximum norm of the sphere for each level
    """
    N = options.N
    alpha = options.alpha
    
    theta_curr = simprior_fn(N, options)
    dists = [1.0]
    d = theta_curr.shape[1]
    
    # Evaluate likelihood at origin for termination condition
    loglike0 = loglike_fn(np.zeros(d), options)
    
    # Initialise
    log_evidence = -np.inf
    levels = [-np.inf]
    terminate = False
    # prod_c_curr = alpha^0 = 1 
    prod_c_curr = 1.0

    # Intialize for tracking
    log_unnormalized_w = []
    log_normalized_w = []
    log_evidence_t = []
    
    # For ranking (Eq (20))
    U = np.random.rand(N) # random numbers for breaking ties
    
    # Compute log likelihoods of N intital particles
    loglike_curr = np.zeros(N)
    for i in range(N):
        loglike_curr[i] = loglike_fn(theta_curr[i, :], options)
        
    count_loglike = N # N evaluations for intial particles
    t = 0 
    
    while not terminate:
        t += 1
        
        # Determine and store threshold choosing by alpha (Algorithm 3 line 2)
        threshL, el_ind, _, _, _ = get_ids(loglike_curr, U, alpha)
        levels.append(threshL)
        
        # Sum the number of particles that are above the threshold
        num_elites = np.sum(el_ind)
        # Calculate the proportion of elite particles (in the slice) \approx adaptation parameter alpha 
        c = num_elites / N
        
        if c == 1.0:
            raise ValueError("All Elites: Too many levels / Particles not moving")
            
        # --- weighted (unnormalized) samples ---
        # reweight eta_{n-1} towards the target distribution pi
        logw = loglike_curr[~el_ind] + np.log(prod_c_curr) # now prod_c_curr = alpha^{t-1} 
        log_unnormalized_w.append(logw)
        # increment for the normalizing constant estimation for pi_{t-1}
        term = logsumexp(logw) - np.log(N) 
        log_normalized_w.append(term)
        # normalizing constant estimation for pi_{t-1}
        log_evidence = np.logaddexp(log_evidence, term)
        log_evidence_t.append(log_evidence)
        
        # now prod_c_curr = alpha^{t} = alpha^{t-1}*alpha
        prod_c_curr = prod_c_curr * c 
        
        if verbose:
            print(f"\nIter {t}\tLevel: {threshL:.4f}\n\t\tCurrent log Z: {log_evidence:.4f}\n\t\tTotal elite: {num_elites}\n")
            
        # --- Resample --- 
        if threshL != np.inf:
            # Find the vector distance of the particle exactly on the threshold
            # Takes the first match [0] in case of exact numerical ties
            match_dist = np.linalg.norm(theta_curr[loglike_curr == threshL][0])
            dists.append(match_dist)
            
            # Note: all valid weights are equal, skip explicit multinomial array and sample uniformly
            # elite_indices = np.where(el_ind)[0]
            # inds = np.random.choice(elite_indices, size=N, replace=True)
            
            # theta_curr = theta_curr[inds, :]
            # loglike_curr = loglike_curr[inds]
            
            # Resample N particles from the uniform distribution on a sphere of radius 'match_dist'
            for i in range(N):
                theta_curr[i, :] = bespoke_exact(match_dist, options.d)
                loglike_curr[i] = loglike_fn(theta_curr[i, :], options)
                count_loglike += 1
                
        # ---- decide whether or not to terminate -----
        if threshL > (np.log(0.75) + loglike0):
            terminate = True
            levels.append(np.inf) # forces final strata
            el_ind = np.zeros(N, dtype=bool)
        # ---------------------------------------------

    # update alpha^{T^N}    
    prod_c_curr = prod_c_curr * c
    
    # reweight eta_{T^N} towards pi
    logw = loglike_curr + np.log(prod_c_curr)
    log_unnormalized_w.append(logw)
    # increment for the normalizing constant estimation for pi_{T^N}
    term = logsumexp(logw) - np.log(N)
    log_normalized_w.append(term)
    # log normalizing constant for pi_{T^N}
    log_evidence = np.logaddexp(log_evidence, term)
    log_evidence_t.append(log_evidence)
    
    return log_evidence, count_loglike, np.array(levels), np.array(dists) # prod_c_curr, np.array(log_unnormalized_w), np.array(log_normalized_w), np.array(log_evidence_t)

In [ ]:
# ANS-SMC with Random Walk (RW) move steps
def ANS_SMC_bespoke_RW(loglike_fn, logprior_fn, simprior_fn, options, verbose=False):
    """
    ANS-SMC with Random Walk (RW) move steps.
    
    Parameters:
    loglike_fn  : callable, log likelihood function taking (sample, options)
    logprior_fn : callable, log prior function taking (sample, options)
    simprior_fn : callable, prior simulation taking (N, options)
    options     : object with attributes `N`, `alpha`, `R`, and `sig`
                  - options.N: Size of population of particles
                  - options.alpha: Threshold so (1-alpha)*N samples are above the next threshold
                  - options.R: Number of MCMC repeats per particle per SMC iteration
                  - options.sig: Standard deviation for mixture RW proposal (sig or sig/4)
    verbose     : bool, set to True to get running updates
    
    Returns:
    log_evidence  : The log evidence estimate
    count_loglike : The total log likelihood computations
    levels        : The levels for nested sampling
    """
    N = options.N
    alpha = options.alpha
    R = options.R
    
    # Draw N particles from the prior
    theta_curr = simprior_fn(N, options)
    d = theta_curr.shape[1]
    
    # Evaluate likelihood at origin for termination condition
    loglike0 = loglike_fn(np.zeros(d), options) # evaluate at the maximum possible log-likelihood
    
    # Initialise
    log_evidence = -np.inf
    t = 0 

    # Intialize for tracking
    log_unnormalized_w = []
    log_normalized_w = []
    log_evidence_t = []
    prop_particles_inside_spike_radius_1sd = []
    prop_particles_inside_spike_radius_2sd = []
    prop_particles_inside_spike_radius_3sd = []
    
    terminate = False
    levels = [-np.inf]
    # prod_c_curr = alpha^0 = 1 
    prod_c_curr = 1.0
    
    # For ranking (Eq (20))
    U = np.random.rand(N)  # random numbers for breaking ties
    
    # Evaluate the log likelihoods of the intial N particles
    loglike_curr = np.zeros(N)
    for i in range(N):
        loglike_curr[i] = loglike_fn(theta_curr[i, :], options)

    count_loglike = N # N evaluations for intial particles
    accept_rate_t = []
    accept_spike_t = []
    accept_slab_t = []

    while not terminate:
        prop_particles_inside_spike_radius_1sd.append(np.sum(np.all(np.abs(theta_curr) < 1 * 0.01, axis=1))/N)
        prop_particles_inside_spike_radius_2sd.append(np.sum(np.all(np.abs(theta_curr) < 2 * 0.01, axis=1))/N)
        prop_particles_inside_spike_radius_3sd.append(np.sum(np.all(np.abs(theta_curr) < 3 * 0.01, axis=1))/N)
        t += 1

        # Determine and store threshold choosing by alpha (Algorithm 3 line 2)
        threshL, el_ind, _, _, _ = get_ids(loglike_curr, U, alpha)
        levels.append(threshL)
        
        # Sum the number of particles that are above the threshold
        num_elites = np.sum(el_ind)
        # Calculate the proportion of elite particles (in the slice) \approx adaptation parameter alpha 
        c = num_elites / N
        
        if c == 1.0:
            raise ValueError("All Elites: Too many levels / Particles not moving")
            
        # --- weighted (unnormalized) samples ---
        # reweight eta_{n-1} towards the target distribution pi
        logw = loglike_curr[~el_ind] + np.log(prod_c_curr) # now prod_c_curr = alpha^{t-1}
        log_unnormalized_w.append(logw)
        # increment for the normalizing constant estimation for pi_{t-1}
        term = logsumexp(logw) - np.log(N) 
        log_normalized_w.append(term)
        # normalizing constant estimation for pi_{t-1}
        log_evidence = np.logaddexp(log_evidence, term)
        log_evidence_t.append(log_evidence)
        
        # now prod_c_curr = alpha^{t} = alpha^{t-1}*alpha
        prod_c_curr = prod_c_curr * c
        
        if verbose:
            print(f"\nIter {t}\tLevel: {threshL:.4f}\n\t\tCurrent log Z: {log_evidence:.4f}\n\t\tTotal elite: {num_elites}\n")
            
        # Resample and Move
        if threshL != np.inf:
            # Note: all valid weights are equal, skip explicit multinomial array and sample uniformly
            # --- Resample ---
            elite_indices = np.where(el_ind)[0]
            inds = np.random.choice(elite_indices, size=N, replace=True) # sample with replacement
            
            # Evaluate log prior for the first time
            if t == 1: 
                logprior_curr = np.zeros(N)
                for i in np.unique(inds):
                    logprior_curr[i] = logprior_fn(theta_curr[i, :], options)

            # Update the current theta, the log likelihoods and priors        
            theta_curr = theta_curr[inds, :]
            loglike_curr = loglike_curr[inds]
            logprior_curr = logprior_curr[inds]
            
            # --- Mutate ---
            accept_R = []
            accept_spike_R = []
            accept_slab_R = []
            prop_spike_R = []   
            prop_slab_R = []

            for k in range(R):
                count_accept = 0
                count_accept_spike = 0
                count_accept_slab = 0
                prop_spike = 0 
                prop_slab = 0

                for i in range(N):
                    in_spike = np.linalg.norm(theta_curr[i, :]) < 3 * 0.01
                    if in_spike:
                        prop_spike += 1
                    else:
                        prop_slab += 1

                    if np.random.rand() < 0.5:
                        sig = options.sig
                    else:
                        sig = options.sig / 4.0
                        
                    theta_prop = np.copy(theta_curr[i, :])
                    loc = np.random.randint(0, d)
                    # Perturb the selected dimension with Gaussian noise
                    theta_prop[loc] += sig * np.random.randn()
                    
                    logprior_prop = logprior_fn(theta_prop, options)
                    
                    # MH acceptance rate
                    if np.exp(logprior_prop - logprior_curr[i]) > np.random.rand():
                        loglike_prop = loglike_fn(theta_prop, options)
                        count_loglike += 1
                        
                        # If the proposed particle has a log likelihood greater than the current threshold threshL, accept it 
                        # and update the current particle
                        if loglike_prop > threshL:
                            theta_curr[i, :] = theta_prop
                            loglike_curr[i] = loglike_prop
                            logprior_curr[i] = logprior_prop
                            count_accept += 1

                            if in_spike:
                                count_accept_spike += 1
                            else: 
                                count_accept_slab += 1

                accept_R.append(count_accept)
                accept_spike_R.append(count_accept_spike)
                accept_slab_R.append(count_accept_slab)
                prop_spike_R.append(prop_spike) 
                prop_slab_R.append(prop_slab)

            total_prop_spike = np.sum(prop_spike_R)
            total_prop_slab = np.sum(prop_slab_R)

            accept_rate_t.append(np.sum(accept_R) / (R * N)) 
            accept_spike_t.append(np.sum(accept_spike_R) / total_prop_spike if total_prop_spike > 0 else np.nan) 
            accept_slab_t.append(np.sum(accept_slab_R) / total_prop_slab if total_prop_slab > 0 else np.nan)               
                            
        # ---- decide whether or not to terminate -----
        if threshL > (np.log(0.75) + loglike0):
            terminate = True
            levels.append(np.inf) # forces final strata
            el_ind = np.zeros(N, dtype=bool)
        # ---------------------------------------------

    # update alpha^{T^N}     
    prod_c_curr = prod_c_curr * c
    
    # reweight eta_{T^N} towards pi
    logw = loglike_curr + np.log(prod_c_curr)
    log_unnormalized_w.append(logw)
    # increment for the normalizing constant estimation for pi_{T^N}
    term = logsumexp(logw) - np.log(N)
    log_normalized_w.append(term)
    # log normalizing constant for pi_{T^N}
    log_evidence = np.logaddexp(log_evidence, term)
    log_evidence_t.append(log_evidence)

    prop_particles_inside_spike_radius_1sd.append(np.sum(np.all(np.abs(theta_curr) < 1 * 0.01, axis=1))/N)
    prop_particles_inside_spike_radius_2sd.append(np.sum(np.all(np.abs(theta_curr) < 2 * 0.01, axis=1))/N)
    prop_particles_inside_spike_radius_3sd.append(np.sum(np.all(np.abs(theta_curr) < 3 * 0.01, axis=1))/N)
    
    return log_evidence, count_loglike, np.array(levels), log_unnormalized_w, log_normalized_w, np.array(log_evidence_t), np.array(prop_particles_inside_spike_radius_1sd), np.array(prop_particles_inside_spike_radius_2sd), np.array(prop_particles_inside_spike_radius_3sd), np.array(accept_rate_t), np.array(accept_spike_t), np.array(accept_slab_t), total_prop_spike, total_prop_slab

In [ ]:
# Diagnostic plots for ANS-SMC 
def plot_ans_smc_diagnostics(
    *,
    levels,
    log_evidence_t,
    prop_1sd,
    prop_2sd,
    prop_3sd,
    accept_rate_t,
    accept_spike_t,
    accept_slab_t,
    true_log_z=np.log(0.3921),
    label_prefix="ANS-SMC",
):
    levels = np.asarray(levels)
    log_evidence_t = np.asarray(log_evidence_t)
    prop_1sd = np.asarray(prop_1sd)
    prop_2sd = np.asarray(prop_2sd)
    prop_3sd = np.asarray(prop_3sd)

    # Align x-axis for threshold-indexed plots (levels often has length T + 1)
    level_x = levels[1:] if len(levels) == len(log_evidence_t) + 1 else levels

    # 1. Running log Z vs. log-likelihood threshold l_t
    plt.figure(figsize=(10, 4.5))
    plt.plot(level_x, log_evidence_t, marker="o", linewidth=1.5, markersize=4, label=r"Running $\log \hat{Z}_t$")
    plt.axhline(
        true_log_z,
        color="r",
        linestyle="--",
        linewidth=1.5,
        label=rf"Ref $\log Z \approx {true_log_z:.5f}$",
    )
    plt.xlabel(r"$l_t$")
    plt.ylabel("Log Evidence")
    plt.title(rf"{label_prefix}: Running Log Evidence vs. $l_t$")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # 2. Running log Z over iterations
    plt.figure(figsize=(10, 4.5))
    plt.plot(log_evidence_t, marker="o", linewidth=1.5, markersize=4, label=r"Running $\log \hat{Z}_t$")
    plt.axhline(
        true_log_z,
        color="r",
        linestyle="--",
        linewidth=1.5,
        label=rf"Ref $\log Z \approx {true_log_z:.5f}$",
    )
    plt.xlabel("Iteration (t)")
    plt.ylabel("Log Evidence")
    plt.title(f"{label_prefix}: Running Log Evidence over Iterations")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # 3. Delta log-likelihood threshold over iterations
    plt.figure(figsize=(10, 4.5))
    plt.plot(np.diff(levels), marker="o", linewidth=1.5, markersize=4)
    plt.xlabel("Iteration (t)")
    plt.ylabel(r"$\Delta l_t$")
    plt.title(f"{label_prefix}: Adaptive Likelihood Step Size Over Iterations")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.show()

    # 4. Acceptance rates over iterations
    plt.figure(figsize=(10, 4.5))
    plt.plot(np.asarray(accept_rate_t) * 100, color="black", linewidth=1.8, label="Overall")
    plt.plot(
        np.asarray(accept_spike_t) * 100,
        color="tab:red",
        linestyle="--",
        linewidth=1.5,
        label="Within 3 SD of Spike",
    )
    plt.plot(
        np.asarray(accept_slab_t) * 100,
        color="tab:blue",
        linestyle="-.",
        linewidth=1.5,
        label="Beyond 3 SD of Spike (Slab)",
    )
    plt.xlabel("Iteration (t)")
    plt.ylabel("Acceptance Rate (%)")
    plt.title(f"{label_prefix}: Acceptance Rates across Iterations")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # 5. Log-likelihood thresholds l_t over iterations
    plt.figure(figsize=(10, 4.5))
    plt.plot(levels, marker="o", linewidth=1.5, markersize=4)
    plt.xlabel("Iteration (t)")
    plt.ylabel(r"$l_t$")
    plt.title(rf"{label_prefix}: $l_t$ Over Time")
    plt.grid(True, linestyle="--")
    plt.tight_layout()
    plt.show()

    # 6. Proportion inside 1 SD vs. threshold l_t
    prop_1sd_x = level_x if len(level_x) == len(prop_1sd) else np.arange(len(prop_1sd))
    plt.figure(figsize=(10, 4.5))
    plt.plot(prop_1sd_x, prop_1sd, marker="o", linewidth=1.5, markersize=4)
    plt.xlabel(r"$l_t$" if len(level_x) == len(prop_1sd) else "Iteration (t)")
    plt.ylabel("Proportion of Particles")
    plt.title(rf"{label_prefix}: Proportion Inside Spike Radius (1 SD) vs. $l_t$")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.show()

    # 7. Running log Z vs. proportion inside 1 SD
    plt.figure(figsize=(10, 4.5))
    plt.plot(prop_1sd, log_evidence_t, marker="o", linewidth=1.5, markersize=4, label=r"Running $\log \hat{Z}_t$")
    plt.axhline(
        true_log_z,
        color="r",
        linestyle="--",
        linewidth=1.5,
        label=rf"Ref $\log Z \approx {true_log_z:.5f}$",
    )
    plt.xlabel("Proportion of Particles (1 SD)")
    plt.ylabel("Log Evidence")
    plt.title(f"{label_prefix}: Running Log Evidence vs. Proportion Inside Spike Radius (1 SD)")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # 8. Proportions inside 1, 2, 3 SD over iterations
    plt.figure(figsize=(10, 4.5))
    plt.plot(prop_1sd, label="1 SD", linewidth=1.5, color="tab:blue")
    plt.plot(prop_2sd, label="2 SD", linestyle="--", linewidth=1.5, color="tab:orange")
    plt.plot(prop_3sd, label="3 SD", linestyle="-.", linewidth=1.5, color="tab:green")
    plt.xlabel("Iteration (t)")
    plt.ylabel("Proportion of Particles")
    plt.title(f"{label_prefix}: Proportion Inside Spike Radius Over Time")
    plt.ylim(-0.05, 1.05)
    plt.legend(loc="upper left")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.show()

    # 9. Proportions inside 1, 2, 3 SD vs. threshold l_t
    plt.figure(figsize=(10, 4.5))
    plt.plot(level_x, prop_1sd, label="1 SD", linewidth=1.5, color="tab:blue")
    plt.plot(level_x, prop_2sd, label="2 SD", linestyle="--", linewidth=1.5, color="tab:orange")
    plt.plot(level_x, prop_3sd, label="3 SD", linestyle="-.", linewidth=1.5, color="tab:green")
    plt.xlabel(r"$l_t$")
    plt.ylabel("Proportion of Particles")
    plt.title(rf"{label_prefix}: Proportion Inside Spike Radius vs. $l_t$")
    plt.ylim(-0.05, 1.05)
    plt.legend(loc="best")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.show()

In [ ]:
# Diagnostic plots for NS-SMC
def _align_levels_x(levels, y):
    levels_arr = np.asarray(levels)
    y_arr = np.asarray(y)
    if len(levels_arr) == len(y_arr):
        return levels_arr
    if len(levels_arr) == len(y_arr) + 1:
        return levels_arr[1:]
    if len(levels_arr) > len(y_arr):
        return levels_arr[-len(y_arr):]
    raise ValueError(
        f"Cannot align levels (len={len(levels_arr)}) with array of length {len(y_arr)}."
    )


def plot_ns_smc_diagnostics(
    *,
    levels,
    log_evidence_t,
    prop_1sd,
    prop_2sd,
    prop_3sd,
    accept_rate_t,
    accept_spike_t,
    accept_slab_t,
    true_log_z=np.log(0.3921),
    label_prefix="NS-SMC",
):
    levels = np.asarray(levels)
    log_evidence_t = np.asarray(log_evidence_t)
    prop_1sd = np.asarray(prop_1sd)
    prop_2sd = np.asarray(prop_2sd)
    prop_3sd = np.asarray(prop_3sd)

    ev_levels_x = _align_levels_x(levels, log_evidence_t)
    prop_levels_x = _align_levels_x(levels, prop_1sd)

    # 1. Running log Z vs. log-likelihood thresholds l_t
    plt.figure(figsize=(10, 4.5))
    plt.plot(ev_levels_x, log_evidence_t, marker="o", linewidth=1.5, markersize=4, label=r"Running $\log \hat{Z}_t$")
    plt.axhline(
        true_log_z,
        color="r",
        linestyle="--",
        linewidth=1.5,
        label=rf"Ref $\log Z \approx {true_log_z:.5f}$",
    )
    plt.xlabel(r"$l_t$")
    plt.ylabel("Log Evidence")
    plt.title(rf"{label_prefix}: Log Evidence vs. $l_t$")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # 2. Running log Z over iterations
    plt.figure(figsize=(10, 4.5))
    plt.plot(log_evidence_t, marker="o", linewidth=1.5, markersize=4, label=r"Running $\log \hat{Z}_t$")
    plt.axhline(
        true_log_z,
        color="r",
        linestyle="--",
        linewidth=1.5,
        label=rf"Ref $\log Z \approx {true_log_z:.5f}$",
    )
    plt.xlabel("Iteration (t)")
    plt.ylabel("Log Evidence")
    plt.title(f"{label_prefix}: Running Log Evidence over Iterations")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # 3. Acceptance rates over iterations
    plt.figure(figsize=(10, 4.5))
    plt.plot(np.asarray(accept_rate_t) * 100, label="Overall", linewidth=1.8, markersize=4, color="black")
    plt.plot(
        np.asarray(accept_spike_t) * 100,
        label="Within 3 SD of Spike",
        linestyle="--",
        linewidth=1.5,
        markersize=4,
        color="tab:red",
    )
    plt.plot(
        np.asarray(accept_slab_t) * 100,
        label="Beyond 3 SD of Spike (Slab)",
        linestyle="-.",
        linewidth=1.5,
        markersize=4,
        color="tab:blue",
    )
    plt.xlabel("Iteration (t)")
    plt.ylabel("Acceptance rate (%)")
    plt.title(f"{label_prefix}: Acceptance Rates across Iterations")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.show()

    # 4. Proportion of particles inside spike radius (1sd, 2sd, 3sd) over iterations
    plt.figure(figsize=(10, 4.5))
    plt.plot(prop_1sd, label="1 SD", linewidth=1.5, markersize=4, color="tab:blue")
    plt.plot(prop_2sd, label="2 SD", linestyle="--", linewidth=1.5, markersize=4, color="tab:orange")
    plt.plot(prop_3sd, label="3 SD", linestyle="-.", linewidth=1.5, markersize=4, color="tab:green")
    plt.xlabel("Iteration (t)")
    plt.ylabel("Proportion of Particles")
    plt.title(f"{label_prefix}: Proportion of Particles Inside Spike Radius Over Time")
    plt.ylim(-0.05, 1.05)
    plt.legend(loc="upper left")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.show()

    # 5. Proportion of particles inside spike radius (1sd, 2sd, 3sd) vs. log-likelihood thresholds
    plt.figure(figsize=(10, 4.5))
    plt.plot(prop_levels_x, prop_1sd, label="1 SD", linewidth=1.5, color="tab:blue")
    plt.plot(prop_levels_x, prop_2sd, label="2 SD", linestyle="--", linewidth=1.5, color="tab:orange")
    plt.plot(prop_levels_x, prop_3sd, label="3 SD", linestyle="-.", linewidth=1.5, color="tab:green")
    plt.xlabel(r"$l_t$")
    plt.ylabel("Proportion of Particles")
    plt.title(rf"{label_prefix}: Proportion of Particles Inside Spike Radius vs. $l_t$")
    plt.ylim(-0.05, 1.05)
    plt.legend(loc="best")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.show()